# PyGeoFetch — 02: Authentication & Providers

This notebook covers the full authentication system and deep-dives into all 22 providers.

---
### What you'll learn
- Add credentials for 12 authenticated providers
- Use environment variables for CI/CD
- Explore all 22 providers and their capabilities
- Filter providers by capability (SAR, VHR, STAC, open access)
- Test and manage stored credentials

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.core.authenticator import AuthManager
from pygeofetch.providers import list_provider_info, get_free_providers, get_providers_by_capability
import pandas as pd

client = PyGeoFetch(log_level="WARNING")
auth = AuthManager()  # Uses 'file' backend by default
print("Ready")

## 1. Adding Credentials

In [ ]:
# ── Method A: Python API ──────────────────────────────────────────────────
# Replace with your real credentials

# USGS Earth Explorer (free account at ers.cr.usgs.gov)
# client.add_credentials("usgs", username="YOUR_USERNAME", password="YOUR_PASSWORD")

# Copernicus CDSE (free at dataspace.copernicus.eu)
# client.add_credentials("copernicus", username="email@example.com", password="YOUR_PASSWORD")

# NASA Earthdata (free at urs.earthdata.nasa.gov) — also used by alaska_satellite_facility
# client.add_credentials("nasa_earthdata", username="USER", password="PASS")
# client.add_credentials("nasa_earthdata_cloud", username="USER", password="PASS")
# client.add_credentials("alaska_satellite_facility", username="USER", password="PASS")

# Planet Labs (api_key from planet.com/account)
# client.add_credentials("planet", api_key="YOUR_PLANET_API_KEY")

# OpenTopography (free tier at portal.opentopography.org)
# client.add_credentials("opentopography", api_key="YOUR_KEY")

# Sentinel Hub (OAuth2 — client credentials from apps.sentinel-hub.com)
# client.add_credentials("sentinel_hub", client_id="YOUR_ID", client_secret="YOUR_SECRET")

# Maxar GBDX (requires Maxar subscription)
# client.add_credentials("maxar_gbdx", api_key="YOUR_MAXAR_TOKEN")

# Airbus OneAtlas (requires Airbus subscription)
# client.add_credentials("airbus_oneatlas", api_key="YOUR_AIRBUS_KEY")

# TerraBotics
# client.add_credentials("terrabotics", api_key="YOUR_KEY")

print("Credential setup cell — fill in YOUR credentials above and uncomment")

In [ ]:
# ── Method B: Environment Variables ──────────────────────────────────────
# Best practice for CI/CD — set these before running
import os

# Show which credential env vars are set
env_prefixes = [
    "PYGEOFETCH_USGS_USERNAME",
    "PYGEOFETCH_COPERNICUS_USERNAME",
    "PYGEOFETCH_PLANET_API_KEY",
    "PYGEOFETCH_NASA_EARTHDATA_USERNAME",
    "PYGEOFETCH_OPENTOPOGRAPHY_API_KEY",
    "PYGEOFETCH_SENTINEL_HUB_CLIENT_ID",
]

print("Credential environment variables:")
for var in env_prefixes:
    status = "✓ SET" if os.environ.get(var) else "✗ not set"
    print(f"  {var:<45} {status}")

In [ ]:
# ── Method C: CLI (run in terminal) ──────────────────────────────────────
# These are the CLI equivalents — run in your terminal, not here

cli_commands = """
# USGS
PyGeoFetch auth add usgs --username USER --password PASS

# Copernicus
PyGeoFetch auth add copernicus --username email@example.com --password PASS

# Planet Labs
PyGeoFetch auth add planet --api-key PL_KEY

# Sentinel Hub
PyGeoFetch auth add sentinel_hub --client-id ID --client-secret SECRET

# Interactive login (prompts for fields)
PyGeoFetch auth login copernicus

# List all stored
PyGeoFetch auth list

# Test credentials
PyGeoFetch auth test usgs
PyGeoFetch auth test copernicus
"""
print(cli_commands)

## 2. Credential Management

In [ ]:
# List all stored credentials
entries = auth.list()
if entries:
    print(f"Stored credentials for {len(entries)} provider(s):")
    for e in entries:
        parts = []
        if e.get("has_username"): parts.append("username")
        if e.get("has_api_key"): parts.append("api_key")
        if e.get("has_token"): parts.append("token")
        print(f"  {e['provider']:<30} [{', '.join(parts)}]")
else:
    print("No credentials stored yet.")
    print("Run the cells above to add credentials for authenticated providers.")

In [ ]:
# Test authentication for each stored provider
# (Only runs if you have credentials stored)
entries = auth.list()
for entry in entries:
    provider = entry["provider"]
    try:
        session = auth.authenticate(provider)
        print(f"✓ {provider}: authenticated (expires {str(session.expires_at)[:19]})")
    except Exception as exc:
        print(f"✗ {provider}: {exc}")

## 3. All 22 Providers — Full Capabilities Table

In [ ]:
# Build comprehensive provider table
providers = list_provider_info()
free_ids = set(get_free_providers())

rows = []
for p in providers:
    rows.append({
        "ID": p["id"],
        "Name": p.get("display_name", p["id"]),
        "Auth": "🌐 Open" if p["id"] in free_ids else f"🔐 {p.get('auth_type', 'required')}",
        "SAR": "✓" if p.get("supports_sar") else "",
        "<1m": "✓" if p.get("supports_sub_meter") else "",
        "STAC": "✓" if p.get("stac") else "",
        "Res Min (m)": p.get("resolution_min_m", ""),
        "Satellites": ", ".join((p.get("satellites") or [])[:3]),
    })

df_all = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 30)
print(f"All {len(df_all)} providers:")
df_all

## 4. Filter Providers by Capability

In [ ]:
# ── SAR providers ─────────────────────────────────────────────────────────
sar_providers = get_providers_by_capability(sar=True)
print(f"SAR providers ({len(sar_providers)}):")
for pid in sar_providers:
    info = next(p for p in providers if p["id"] == pid)
    auth_label = "🌐" if pid in free_ids else "🔐"
    print(f"  {auth_label} {pid:<35} {info.get('display_name', '')}")

In [ ]:
# ── Sub-metre (<1m) VHR providers ─────────────────────────────────────────
vhr_providers = get_providers_by_capability(sub_meter=True)
print(f"Sub-metre VHR providers ({len(vhr_providers)}):")
for pid in vhr_providers:
    info = next(p for p in providers if p["id"] == pid)
    res_min = info.get("resolution_min_m", "?")
    print(f"  🔐 {pid:<35} Best resolution: {res_min}m")

In [ ]:
# ── STAC-compliant providers ───────────────────────────────────────────────
stac_providers = get_providers_by_capability(stac=True)
print(f"STAC 1.0 providers ({len(stac_providers)}):")
for pid in stac_providers:
    info = next(p for p in providers if p["id"] == pid)
    auth_label = "🌐 open" if pid in free_ids else "🔐 auth"
    print(f"  {pid:<35} {auth_label}   {info.get('endpoint_url', '')}")

In [ ]:
# ── Open access only (no login) ───────────────────────────────────────────
open_providers = get_providers_by_capability(requires_auth=False)
print(f"Open access providers ({len(open_providers)}):")
for pid in open_providers:
    info = next(p for p in providers if p["id"] == pid)
    tags = []
    if info.get("stac"): tags.append("STAC")
    if info.get("supports_sar"): tags.append("SAR")
    if info.get("supports_sub_meter"): tags.append("<1m")
    tag_str = f"[{', '.join(tags)}]" if tags else ""
    print(f"  🌐 {pid:<35} {tag_str}")

## 5. Provider Deep Dives

In [ ]:
# Get full capabilities for any provider
from pygeofetch.providers import get_provider

for pid in ["planetary_computer", "aws_earth", "copernicus", "planet"]:
    p = get_provider(pid)
    caps = p.get_capabilities()
    quota = p.get_quota_info()
    
    print(f"\n{'='*60}")
    print(f"  {caps.name} ({pid})")
    print(f"{'='*60}")
    print(f"  Auth type:    {caps.auth_type}")
    print(f"  Requires auth:{caps.requires_auth}")
    print(f"  SAR support:  {caps.supports_sar}")
    print(f"  Sub-metre:    {caps.supports_sub_meter}")
    print(f"  STAC:         {caps.stac}")
    print(f"  CQL2:         {caps.supports_cql2}")
    print(f"  Resolution:   {caps.resolution_min_m}m – {caps.resolution_max_m}m")
    print(f"  Satellites:   {', '.join(caps.satellites[:5])}")
    print(f"  Docs:         {caps.docs_url}")
    print(f"  Endpoint:     {caps.endpoint_url}")

## 6. Status Dashboard

In [ ]:
# Full status: which providers are ready to use right now
status = client.status()
authed = set(status["providers_authenticated"])
free = set(status["providers_free"])

print(f"PyGeoFetch v{status['version']}")
print(f"Cache entries: {status['cache_entries']}")
print()

print(f"{'Provider':<35} {'Status':<25} {'Can Search?'}")
print("-" * 70)
for p in list_provider_info():
    pid = p["id"]
    if pid in authed:
        status_str = "✓ authenticated"
        can_search = "Yes"
    elif pid in free:
        status_str = "🌐 open (no auth needed)"
        can_search = "Yes"
    else:
        status_str = "✗ credentials needed"
        can_search = "No"
    print(f"{pid:<35} {status_str:<25} {can_search}")

---
## Summary
- ✅ Added credentials via Python API, environment variables, and CLI
- ✅ Explored all 22 providers and their capabilities
- ✅ Filtered providers by SAR, VHR, STAC, and open access
- ✅ Built a live status dashboard

**Next:** `03_advanced_search.ipynb` — multi-provider search, CQL2 filters, geometry files, and output formats.